# High score range

Анализ яркой части распределения скора для выбранного чекпоинта.
Кэш — в `cached_data/<CHECKPOINT>/`.


In [ ]:
CHECKPOINT = ("261002_0535_da_nu_classifier_exp_full_E1_lambda0.01_DArepair"
              "@best_da_model")
FORCE_RELOAD = False   # True = пересобрать кэш из источника


In [ ]:
import os, sys
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
ROOT = "/home/albert/Baikal2025"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + "/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")
os.chdir("/home/albert/Baikal2025/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")  # относительные пути как в исходных ноутбуках

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from inference import reader
from excess.high import load_or_build, fit_domain, with_fit, DOMAINS, COLOURS
from excess.features import add_derived_columns

pd.set_option("display.width", 200)

EV, FIT, report, origin = load_or_build(CHECKPOINT, force_reload=FORCE_RELOAD)
print(f"источник данных: {origin}")
display(report)

add_derived_columns(EV)
muatm = EV["muatm"].join(FIT["muatm"])
exp = EV["exp_full"].join(FIT["exp_full"])
nuatm = EV["nuatm"]
nue2 = EV["nue2"]
hist_kwargs = dict(density=True, bins=50, alpha=0.5, histtype="step")


In [ ]:
EV['exp_full'][EV['exp_full']['score'] > 0.7]['part_key'].value_counts().sort_index()

In [ ]:
# def _apply_mask(df: pd.DataFrame):
#     mask = df.n_sn_hits>=8
#     mask &= df.n_sn_strings>=3
#     if 'fit_status' in df.columns:
#         mask &= df.fit_status == 'ok'
#     return df[mask].copy()

# muatm_masked = _apply_mask(muatm)
# exp_masked = _apply_mask(exp)
# nue2_masked = _apply_mask(nue2)
# nuatm_masked = _apply_mask(nuatm)

# for col_to_plot, range_tuple in [
#     ("z_verticality_1", (0, 3)), 
#     ("z_t_slope_m_per_ns", None),
#     ("z_t_slope_error_m_per_ns", None),
#     ("z_t_slope_m_per_ns_q", None),
#     ("z_t_residual_rms_m", None),
#     ("z_t_residual_rms_m_q", None),
#     ("sigma_z_m", (0, 50)),
#     ("sigma_z_m_q", (0, 50)),
#     ]:
#     _hist_kwargs = dict(density=True, bins=50, alpha=0.5, histtype='step', log=True, range=range_tuple)
#     plt.hist(muatm_masked[col_to_plot].clip(*range_tuple) if range_tuple else muatm_masked[col_to_plot], label=f"muatm, total={len(muatm_masked)}", **_hist_kwargs)
#     plt.hist(exp_masked[col_to_plot].clip(*range_tuple) if range_tuple else exp_masked[col_to_plot], label=f"exp, total={len(exp_masked)}", **_hist_kwargs)
#     plt.hist(nue2_masked[col_to_plot].clip(*range_tuple) if range_tuple else nue2_masked[col_to_plot], label=f"nue2, total={len(nue2_masked)}", ls='--', **_hist_kwargs)
#     plt.xlabel(col_to_plot)
#     plt.ylabel("Density")
#     plt.legend()
#     plt.grid(ls=":")
#     plt.show()

In [ ]:
def hitsstrings_mask(df: pd.DataFrame):
    mask = (df.n_sn_hits>=8)
    mask &= (df.n_sn_strings>=3)
    return mask

def _apply_mask(df: pd.DataFrame):
    #mask = (df.z_t_slope_m_per_ns_q >= 0.0)
    #mask &= (df.logit>=4)
    #mask = (df.sigma_z_m >= 50)
    #mask &= (df.z_verticality_1 <= 4000)
    #mask &= (df.z_t_residual_rms_m_q <= 15)
    #mask &= (df.z_t_slope_error_m_per_ns_q <= 0.1)
    #mask &= (df.n_hits < 80)
    #mask &= (df.n_hits >= 50) & (df.n_hits <= 100)
    #mask = (df.n_sn_hits>=8)
    #mask &= (df.n_sn_strings>=3)
    mask = (df.score>=0.8)
    mask &= (df.n_sn_hits>=8)
    mask &= (df.n_sn_strings>=3)


    # mask &= (df.fit_n_hits>=8)
    # mask &= (df.fit_n_strings>=3)
    mask &= (df.fit_status == 'ok')
    # mask &= (df.fit_cov_matrix_status == 3)
    
    
    # mask &= (df.fit_z_dist > 70)
    # # mask &= (df.fit_scf_max_theta < 1.7)
    # mask &= (np.log10(df["fit_time_chi2"] / (df.fit_n_hits - 5)) < 1.0)

    # mask &= ((df["fit_n_triplets"] / (df.fit_n_hits - 5)) > 0.1)
    # mask &= ((df["fit_q_acc_20"] / df["fit_q_tot_20"]) > 0.05)
    # mask &= (df.fit_t_res_30_med < 60)
    # mask &= (df["fit_log_phit_30_min"] > -9)
    # mask &= (df.fit_theta_err <= 2.5)

    # mask &= (df["fit_ev_center_z"] < 220)
    # mask &= (df.fit_theta_rec <= 80)

    return df[mask].copy()


col_to_plot = "logit"
x_range = (0.0, 5.5)
_hist_kwargs = dict(density=True, bins=50, alpha=0.5, histtype='step', log=True, range=x_range)
plt.hist(_apply_mask(muatm)[col_to_plot], label=f"muatm, total={len(_apply_mask(muatm))} / {sum(hitsstrings_mask(muatm))}={len(_apply_mask(muatm))/sum(hitsstrings_mask(muatm))*100 if sum(hitsstrings_mask(muatm)) > 0 else 0:.3f}%", **_hist_kwargs)
plt.hist(_apply_mask(exp)[col_to_plot], label=f"exp, total={len(_apply_mask(exp))} / {sum(hitsstrings_mask(exp))}={len(_apply_mask(exp))/sum(hitsstrings_mask(exp))*100 if sum(hitsstrings_mask(exp)) > 0 else 0:.3f}%", **_hist_kwargs)
#plt.hist(_apply_mask(nue2)[col_to_plot], label=f"nue2, total={len(_apply_mask(nue2))} / {sum(hitsstrings_mask(nue2))}={len(_apply_mask(nue2))/sum(hitsstrings_mask(nue2))*100 if sum(hitsstrings_mask(nue2)) > 0 else 0:.3f}%", ls='--', **_hist_kwargs)
#plt.hist(_apply_mask(nuatm)[col_to_plot], label=f"nuatm, total={len(_apply_mask(nuatm))} / {sum(hitsstrings_mask(nuatm))}={len(_apply_mask(nuatm))/sum(hitsstrings_mask(nuatm))*100 if sum(hitsstrings_mask(nuatm)) > 0 else 0:.3f}%", ls='--', **_hist_kwargs)
plt.xlabel(col_to_plot)
plt.ylabel("Density")
plt.legend()
plt.grid(ls=":")
plt.xlim(*x_range)
plt.show()

In [ ]:
# np.log10(muatm['fit_time_chi2']/ (muatm.fit_n_hits - 5)).hist(bins=50, alpha=0.5, histtype='step', density=True, range=(-1, 2))
# np.log10(exp['fit_time_chi2']/ (exp.fit_n_hits - 5)).hist(bins=50, alpha=0.5, histtype='step', density=True, range=(-1, 2))
# plt.show()

# muatm['fit_z_dist'].hist(bins=50, alpha=0.5, histtype='step', density=True, range=(0, 200))
# exp['fit_z_dist'].hist(bins=50, alpha=0.5, histtype='step', density=True, range=(0, 200))

In [ ]:
# muatm_residual = _apply_mask(muatm)
# (muatm_residual['prime_theta_deg'] - muatm_residual['fit_theta_rec']).hist(bins=50, alpha=0.5, histtype='step')
# muatm_residual['fit_theta_err'].hist(bins=50, alpha=0.5, histtype='step')

In [ ]:
from excess import display as events_drawer
from inference.track_fit_alike_bars import geometry, isolation
from inference.track_fit_alike_bars.physics import propagation_time

ADDRESS = ["source", "part_key", "local_idx"]


def reconstruction_cuts(frame):
    """Каты качества реконструкции -- тот же набор, что и в остальных ячейках."""
    return ((frame.fit_status == "ok") & (frame.fit_n_hits >= 8)
            & (frame.fit_n_strings >= 3) & (frame.fit_cov_matrix_status == 3)
            & (frame.fit_z_dist > 70)
            & (np.log10(frame.fit_time_chi2 / (frame.fit_n_hits - 5)) < 1.0)
            & (frame.fit_n_triplets / (frame.fit_n_hits - 5) > 0.1)
            & (frame.fit_q_acc_20 / frame.fit_q_tot_20 > 0.05)
            & (frame.fit_t_res_30_med < 60) & (frame.fit_log_phit_30_min > -9)
            & (frame.fit_theta_err <= 2.5) & (frame.fit_ev_center_z < 220)
            & (frame.fit_theta_rec <= 80))


def track_of(row, *, prefix="fit_"):
    """Трек строки словарём (`ref_x`...`dir_z`, `ref_time`) для `isolation.*`.

    None, если фит не сошёлся или его параметров в строке нет: у такой строки нет
    трека, и всё, что считается от него, тоже не определено.
    """
    if row.get(f"{prefix}status") != "ok":
        return None
    track = {}
    for name in isolation.TRACK_FIELDS + ("ref_time",):
        value = row.get(f"{prefix}{name}", np.nan)
        if not np.isfinite(value):
            return None
        track[name] = float(value)
    return track


class Selection:
    """Выбранные события, прочитанные один раз: таблицы и картинка из одного чтения.

    `display.load` читает хиты, прогоняет фит и собирает маски; сюда добавлены
    геометрия (позиции всех ОМ, включая молчащие) и изоляция -- `pnonhit` и то, из
    чего он складывается.  Кадру нужен адрес события (`source`, `part_key`,
    `local_idx`, для МС ещё `data_class`); колонки `fit_*` -- только для `pnonhit`
    и `t_res`.

        sel = Selection(exp[reconstruction_cuts(exp)].nlargest(8, "score"))
        sel.pnonhit()      # кадр + iso_pnonhit, iso_n_non_hit, iso_n_oms, iso_min_p_silent
        sel.hits           # строка -- импульс: маски, геометрия трека, p_fire
        sel.empty_oms()    # пустые ОМ вокруг трека: из них складывается pnonhit
        sel.figure(name="exp_top8")
    """

    def __init__(self, frame, *, prefix="fit_", **load_kwargs):
        # индекс входа сохраняется: `event_fk` в таблицах -- это он, и по нему
        # таблицы стыкуются обратно с кадром, из которого выборка сделана
        self.frame = frame
        self.prefix = prefix
        self.events = events_drawer.load(frame.reset_index(drop=True), **load_kwargs)
        self._pnonhit = None

    def pnonhit(self):
        """Кадр событий плюс `iso_*` -- обёртка над `isolation.attach`.

        Событие без сошедшегося фита получает NaN: трека нет, судить нечего.
        Результат запоминается, и `hits` потом добавляет эти колонки к хитам.
        """
        self._pnonhit = isolation.attach(self.frame, fit_prefix=self.prefix)
        return self._pnonhit

    @property
    def hits(self):
        """Строка -- импульс: заряд, время, координаты, канал, маски, геометрия.

        `is_sig` -- сеть сигнал/шум, `is_fit` -- трек реконструкции, `is_truth*` --
        правда МС.  От трека: `dist_perp` и `dist_light` (м, перпендикулярное и
        световое расстояние до линии), `p_fire` (вероятность срабатывания этого ОМ
        по световой модели) и `t_res` (невязка времени, нс).  `event_fk` -- индекс
        строки в кадре выборки, по нему таблица стыкуется обратно.

        `p_fire` и `t_res` осмысленны вблизи трека: у шумового хита в сотне метров
        они тоже посчитаны, но говорят лишь о том, что модель света его не ждала.
        """
        source = self._pnonhit if self._pnonhit is not None else self.frame
        blocks = []
        for (key, row), event in zip(source.iterrows(), self.events):
            block = pd.DataFrame({"q": event.q, "t": event.t, "x": event.x,
                                  "y": event.y, "z": event.z,
                                  "channel": np.asarray(event.channel)})
            for name, mask in event.masks.items():
                block[name] = mask
            track = track_of(row, prefix=self.prefix)
            if track is not None:
                om = geometry.oms(event.source, event.part_key,
                                  local_idx=int(event.local_idx))
                channel = np.asarray(event.channel, dtype=np.int64)
                block["dist_perp"] = isolation.distances(track, om)[channel]
                block["dist_light"] = isolation.distances(
                    track, om, distance="light")[channel]
                block["p_fire"] = isolation.phit_field(track, om)[channel]
                ref = np.array([track["ref_x"], track["ref_y"], track["ref_z"]])
                direction = np.array([track["dir_x"], track["dir_y"], track["dir_z"]])
                block["t_res"] = event.t - track["ref_time"] - np.array(
                    [propagation_time(*ref, *direction, x, y, z)
                     for x, y, z in zip(event.x, event.y, event.z)])
            block["event_fk"] = key
            for column in ("score", "n_sn_hits", "fit_n_hits", "fit_theta_rec",
                           "iso_pnonhit", "iso_n_non_hit", "iso_n_oms"):
                if column in row.index:
                    block[column] = row[column]
            blocks.append(block)
        return pd.concat(blocks, ignore_index=True)

    def empty_oms(self, *, radius=isolation.RADIUS):
        """Пустые ОМ вокруг трека -- то, из чего складывается `pnonhit`.

        Строка -- модуль: адрес события, канал, координаты в кадре хитов,
        расстояние до трека и вероятность сработать.  `pnonhit` события -- их
        геометрическое среднее, а минимальная `p_fire` -- тот модуль, срабатывания
        которого трек ждал сильнее всего.  Пусто там, где трека нет или в окне не
        осталось пустых модулей.
        """
        rows = []
        for (key, row), event in zip(self.frame.iterrows(), self.events):
            track = track_of(row, prefix=self.prefix)
            if track is None:
                continue
            om = geometry.oms(event.source, event.part_key,
                              local_idx=int(event.local_idx))
            dist = isolation.distances(track, om, distance="light")
            fired = np.asarray(event.channel, dtype=np.int64)
            empty = om.usable() & (dist <= radius) & ~np.isin(om.channel, fired)
            if not empty.any():
                continue
            rows.append(pd.DataFrame({
                "event_fk": key, "part_key": event.part_key,
                "local_idx": event.local_idx, "channel": om.channel[empty],
                "x": om.xyz[empty, 0], "y": om.xyz[empty, 1], "z": om.xyz[empty, 2],
                "dist_light": dist[empty],
                "p_fire": isolation.phit_field(track, om)[empty]}))
        if not rows:
            return pd.DataFrame(columns=["event_fk", "part_key", "local_idx", "channel",
                                         "x", "y", "z", "dist_light", "p_fire"])
        return pd.concat(rows, ignore_index=True)

    def figure(self, *, name=None, dpi=200, notes=None, **kwargs):
        """Картинка по строке на событие -- `display.figure`, как в остальных ячейках.

        `name` заодно сохраняет png и сайдкар провенанса в `figures/`.
        """
        figure = events_drawer.figure(self.events, **kwargs)
        if name:
            events_drawer.save(figure, name=name, dpi=dpi, notes=notes)
        return figure

In [ ]:
# События для показа: сошедшиеся, прошедшие каты реконструкции, сильнейшие по скору.
# Дальше -- любые ваши условия; важно только, чтобы кадр нёс адрес события.
take = exp[reconstruction_cuts(exp)].nlargest(8, "score")
sel = Selection(take)

sel.pnonhit()[["score", "fit_n_hits", "fit_theta_rec",
               "iso_pnonhit", "iso_n_non_hit", "iso_n_oms", "iso_min_p_silent"]]

In [ ]:
# Хиты -- одна таблица на все выбранные события: разрезы, выбросы, что угодно.
sel.hits[["event_fk", "q", "t", "dist_light", "p_fire", "t_res",
          "is_sig", "is_fit"]].head(8)

In [ ]:
sel.empty_oms()

In [ ]:
# Строка кадра -- строка картинки.  `name` сохраняет png в figures/ с сайдкаром.
sel.figure(name="exp_distinct_top8")

In [ ]:
# Так же -- любой другой домен: у МС в панели появятся правда и треки мюонов,
# а `fit_*`-колонки нужны только для `pnonhit` и `t_res`.
take = muatm[reconstruction_cuts(muatm)].nlargest(6, "score")
Selection(take).figure(name="muatm_distinct_top6")